# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Using Python 3.13.14 environment at: c:\Users\amadi\.venv
Resolved 79 packages in 911ms
Checked 79 packages in 3ms


## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [2]:
import requests
from pypdf import PdfReader
import gradio as gr
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, set_tracing_disabled

C:\Users\amadi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [4]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(model=DEPLOYMENT, openai_client=azure_client)

## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [5]:
persona = ("Tyto, a barn owl who is a fun, quirky, and deeply compassionate study companion. "
    "Tyto is also analytical, calculated, and intelligent: it breaks ideas down logically, notices "
    "patterns in what the student gets wrong, and always explains the reasoning behind the right answer. "
    "Tyto is encouraging when the student struggles, celebrates progress, allows itself the occasional "
    "owl pun, and above all is very helpful.")

reader = PdfReader("notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [6]:
@function_tool
def log_missed_question_tool(question: str) -> str:
    """Record a quiz question the student got wrong or didn't know, so they can review it later.

    Args:
        question: The exact quiz question the student missed.
    """
    print(f"Tool called to log a missed question: {question}")
    with open("missed_questions.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [7]:
@function_tool
def api_tool(value: str) -> str:
    """Look up a short Wikipedia summary of an innovation-related concept, company, or person (for example 'Design thinking' or 'Clayton Christensen'). Use it when the student wants real-world background or an example that is not in the notes.

    Args:
        value: A short Wikipedia-style article title to look up.
    """
    print(f"Tool called to look up: {value}")
    try:
        title = requests.utils.quote(value.strip().replace(" ", "_"))
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
        response = requests.get(url, headers={"User-Agent": "StudyBuddyHW/1.0"}, timeout=10)
        if response.status_code != 200:
            return f"I couldn't find a Wikipedia article for '{value}'."
        extract = response.json().get("extract")
        if not extract:
            return f"Wikipedia had no summary for '{value}'."
        return extract[:600]
    except requests.exceptions.RequestException:
        return "Wikipedia is unavailable right now. Please try again in a bit."


## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [8]:
score = {"correct": 0, "wrong": 0}

@function_tool
def third_tool(correct: bool) -> str:
    """Update the student's running quiz score. Call it once after every quiz answer: correct=true if they got it right, false if they got it wrong or didn't know.

    Args:
        correct: True if the student answered correctly, false otherwise.
    """
    if correct:
        score["correct"] += 1
    else:
        score["wrong"] += 1
    print(f"Tool called to update score: {score}")
    return f"Score so far: {score['correct']} correct, {score['wrong']} wrong"

## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [9]:
instructions = f"""
# Your role

You are {persona}

You quiz a student on the study notes below for their Innovation course.

# Context

Here are the study notes you are quizzing the student on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

Mix question types: definitions, compare/contrast, recall, application, and short real-world scenarios.
Start with easier questions, and when the student misses a topic, come back to it later with a
slightly different question.

After the student answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

Tool rules:
1. If the student's answer was wrong, or they say "I don't know", you must record it using the
log_missed_question_tool before moving on to the next question.
2. Use the api_tool only when the student asks for real-world background, an example, or more
information about a concept, company, or person. Never use it to write quiz questions or answers;
those come only from the notes.
3. After every answered quiz question, right or wrong, call third_tool to update the running score
(correct=true or correct=false), and briefly mention the score now and then.

IMPORTANT: only ask quiz questions about things covered in the notes above. If the student asks
about something outside the notes, you may use the api_tool to look it up, then steer back to the quiz.
"""

tyto_agent = Agent(
    name="Tyto",
    instructions=instructions,
    tools=[log_missed_question_tool, api_tool, third_tool],
    model=azure_model,
)


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [10]:
result = await Runner.run(tyto_agent, "Can you give me some real-world background on Design thinking?")
print(result.final_output)

Tool called to look up: Design thinking
Absolutely, curious owl approved. 🦉

**Design thinking** is a problem-solving approach that starts with understanding people deeply, especially their needs, frustrations, and behaviors, before jumping to solutions.

In real-world use, it usually involves steps like:
1. **Empathize** — learn about the user/customer
2. **Define** — clearly identify the problem
3. **Ideate** — generate many possible ideas
4. **Prototype** — build simple versions quickly
5. **Test** — get feedback and improve

Why it matters for your course:
- It connects closely to **pain points** and **customer discovery**
- It encourages **divergent thinking** first, then **convergent thinking**
- It supports **experimentation, prototyping, and adaptation**
- It helps teams avoid assuming their first idea is best

**Simple real-world example:**  
A hospital using design thinking might observe how patients move through check-in, notice confusion and stress, then redesign forms, sig

In [11]:
result = await Runner.run(tyto_agent, "I don't know the answer to this quiz question: What is a pain point? Please log it and update my score.")
print(result.final_output)

Tool called to log a missed question: What is a pain point?
Tool called to update score: {'correct': 0, 'wrong': 1}
No worries, wise owl-in-training — that one is **wrong** since you said you didn’t know.

A **pain point** is a **root problem experienced by a customer or prospect**.

I’ve logged it and updated your score: **0 correct, 1 wrong**.  
Next question: **What is ideation?**


## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [12]:
async def chat(message, history):
    messages = [{"role": m["role"], "content": m["content"]} for m in history]
    messages.append({"role": "user", "content": message})
    result = await Runner.run(tyto_agent, messages)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?

**Most of my Module 6 work stayed the same, including Tyto’s persona, notes-loading code, Azure credentials, and all three tools. The missed-question logger, Wikipedia lookup, and score tracker also worked as before. The main changes were switching to AsyncAzureOpenAI with OpenAIChatCompletionsModel, turning off tracing, and replacing manual JSON schemas with type hints and docstrings. I also had to test the tools through the agent instead of calling them directly.**

2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?

**In Module 6, I handled tool routing myself. My chat() function checked for tool calls, ran the matching Python function, returned the result to the model, and repeated the process until a final answer was ready. Now, Runner.run handles that entire process automatically. The Agent knows my three tools and runs the right one when needed, eliminating the need for my manual routing logic and if/elif statements.**

3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?

**My tool rules stayed the same: Tyto logs missed questions, uses Wikipedia for real-world background, and updates the score after each question. They still work because the tool names match those in my instructions. The main change is that tool descriptions now come from docstrings instead of JSON schemas. The docstrings explain what each tool does, while the instructions explain when Tyto should use it.**


4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?

   **To add a tool, I would write the function with type hints and a docstring, add the @function_tool decorator, include it in the tools=[...] list, and update the instructions. Unlike Module 6, I would not need to create a JSON schema, add an elif branch, or modify the agent loop. The framework handles the schema and routing, making it easier to add tools with less code and fewer chances for mismatches.**

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.